# 🎓 04 - Handle-Pull ACT Training

This notebook trains an ACT policy for MetaWorld `handle-pull-v3`.

**Key Steps:**
1. Install dependencies
2. 🔍 DEBUG: Verify dataset
3. Train ACT policy
4. 🔍 DEBUG: Test policy outputs
5. Evaluate policy

In [ ]:
# ==========================================
# CELL 1: SYSTEM DEPENDENCIES
# ==========================================

!apt-get update -qq
!apt-get install -y -qq libgl1-mesa-dev libgl1-mesa-glx libglew-dev \
                         libosmesa6-dev software-properties-common patchelf

print("✅ System dependencies installed")

In [ ]:
# ==========================================
# CELL 2: ENVIRONMENT SETUP
# ==========================================

import os
import sys

os.environ['MUJOCO_GL'] = 'egl'
os.environ['LEROBOT_VIDEO_BACKEND'] = 'pyav'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
    print("✅ Secrets loaded")
except:
    print("⚠️ Set secrets manually")

import wandb
wandb.login(key=os.environ.get('WANDB_API_KEY', ''))
print("✅ Environment configured")

In [ ]:
# ==========================================
# CELL 3: INSTALL PACKAGES
# ==========================================

!git clone https://github.com/huggingface/lerobot.git /kaggle/working/lerobot 2>/dev/null || echo "Already cloned"
%cd /kaggle/working/lerobot
!pip install -e . -q
!pip install metaworld wandb imageio imageio-ffmpeg av -q

print("\n✅ All packages installed")

In [ ]:
# ==========================================
# CELL 4: CONFIGURATION
# ==========================================

TASK_NAME = "handle-pull-v3"
HF_USERNAME = "aryannzzz"

DATASET_REPO_ID = f"{HF_USERNAME}/metaworld-{TASK_NAME}-expert-v2"

TRAINING_STEPS = 100000
BATCH_SIZE = 8
LEARNING_RATE = 0.0001
CHUNK_SIZE = 20  # Small to prevent mode collapse

OUTPUT_DIR = "/kaggle/working/outputs/handlepull"
POLICY_REPO_ID = f"{HF_USERNAME}/act-handle-pull-v2"

print(f"📋 Task: {TASK_NAME}")
print(f"📋 Dataset: {DATASET_REPO_ID}")
print(f"�� Chunk size: {CHUNK_SIZE}")

---
## 🔍 DEBUG: Verify Dataset

In [ ]:
# ==========================================
# CELL 5: 🔍 DEBUG - Load Dataset
# ==========================================

import sys
sys.path.insert(0, "/kaggle/working/lerobot/src")

from lerobot.datasets.lerobot_dataset import LeRobotDataset
import numpy as np
import torch

print(f"🔍 Loading: {DATASET_REPO_ID}")

dataset = LeRobotDataset(DATASET_REPO_ID)
print(f"\n✅ Loaded: {len(dataset)} frames, {dataset.num_episodes} episodes")

sample = dataset[0]
print(f"\n📐 Sample:")
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f"   {k}: {v.shape}")

In [ ]:
# ==========================================
# CELL 6: 🔍 DEBUG - Check Actions
# ==========================================

all_actions = [dataset[i]['action'].numpy() for i in range(min(500, len(dataset)))]
all_actions = np.array(all_actions)

print(f"📊 Action Stats:")
print(f"   Mean: {all_actions.mean(axis=0)}")
print(f"   Std:  {all_actions.std(axis=0)}")

if np.any(all_actions.std(axis=0) < 0.1):
    print("\n⚠️ Low variance detected!")
else:
    print("\n✅ Actions look good!")

---
## 🎓 Training

In [ ]:
# ==========================================
# CELL 7: TRAIN ACT POLICY
# ==========================================

!python /kaggle/working/lerobot/src/lerobot/scripts/lerobot_train.py \
    --policy.type=act \
    --policy.repo_id={POLICY_REPO_ID} \
    --env.type=metaworld \
    --env.task={TASK_NAME} \
    --dataset.repo_id={DATASET_REPO_ID} \
    --steps={TRAINING_STEPS} \
    --batch_size={BATCH_SIZE} \
    --optimizer.lr={LEARNING_RATE} \
    --eval_freq=-1 \
    --save_freq=25000 \
    --log_freq=100 \
    --policy.chunk_size={CHUNK_SIZE} \
    --policy.n_obs_steps=1 \
    --wandb.enable=true \
    --wandb.project=metaworld-act-handlepull \
    --output_dir={OUTPUT_DIR}

---
## 🧪 Evaluation

In [ ]:
# ==========================================
# CELL 8: LOAD POLICY
# ==========================================

import torch
from pathlib import Path
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.factory import make_pre_post_processors
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoints = sorted(Path(OUTPUT_DIR).glob("checkpoint_*"))
POLICY_CHECKPOINT = checkpoints[-1] if checkpoints else POLICY_REPO_ID
print(f"📁 Using: {POLICY_CHECKPOINT}")

policy = ACTPolicy.from_pretrained(str(POLICY_CHECKPOINT))
policy = policy.to(device).eval()
print(f"✅ Policy loaded on {device}")

dataset_meta = LeRobotDatasetMetadata(DATASET_REPO_ID)
preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy.config,
    dataset_stats=dataset_meta.stats,
)
print("✅ Processors ready")

In [ ]:
# ==========================================
# CELL 9: 🔍 DEBUG - Test Policy Outputs
# ==========================================

from lerobot.envs.metaworld import MetaworldEnv

print("🔍 Testing policy outputs...")

env = MetaworldEnv(
    task=TASK_NAME,
    obs_type="pixels_agent_pos",
    render_mode="rgb_array",
    observation_width=480,
    observation_height=480,
)

obs, _ = env.reset(seed=0)
policy.reset()

policy_actions = []
for _ in range(20):
    batch = {}
    img = torch.from_numpy(obs['pixels']).float() / 255.0
    img = img.permute(2, 0, 1).unsqueeze(0).to(device)
    batch['observation.images.image'] = img
    
    state = torch.from_numpy(obs['agent_pos']).float().unsqueeze(0).to(device)
    batch['observation.state'] = state
    
    processed = preprocessor(batch)
    with torch.no_grad():
        action = policy.select_action(processed)
    action = postprocessor(action)
    action_np = action.cpu().numpy().squeeze()
    policy_actions.append(action_np)
    obs, _, _, _, _ = env.step(action_np)

env.close()

policy_actions = np.array(policy_actions)
print(f"\n📊 Policy: Mean={policy_actions.mean(axis=0)}, Std={policy_actions.std(axis=0)}")
print(f"📊 Dataset: Mean={all_actions.mean(axis=0)}, Std={all_actions.std(axis=0)}")

ratio = all_actions.std(axis=0) / (policy_actions.std(axis=0) + 1e-6)
if np.any(ratio > 5):
    print("\n❌ MODE COLLAPSE!")
else:
    print("\n✅ Outputs OK!")

In [ ]:
# ==========================================
# CELL 10: EVALUATE
# ==========================================

print("🧪 Evaluating (10 episodes)...")

env = MetaworldEnv(
    task=TASK_NAME,
    obs_type="pixels_agent_pos",
    render_mode="rgb_array",
    observation_width=480,
    observation_height=480,
)

successes = 0
for ep in range(10):
    obs, _ = env.reset(seed=ep)
    policy.reset()
    
    for step in range(300):
        batch = {}
        img = torch.from_numpy(obs['pixels']).float() / 255.0
        img = img.permute(2, 0, 1).unsqueeze(0).to(device)
        batch['observation.images.image'] = img
        batch['observation.state'] = torch.from_numpy(obs['agent_pos']).float().unsqueeze(0).to(device)
        
        processed = preprocessor(batch)
        with torch.no_grad():
            action = policy.select_action(processed)
        action = postprocessor(action).cpu().numpy().squeeze()
        
        obs, _, term, trunc, info = env.step(action)
        
        if info.get('success', False) or info.get('is_success', False):
            successes += 1
            print(f"   Ep {ep+1}: ✅ SUCCESS")
            break
        if term or trunc:
            print(f"   Ep {ep+1}: ❌ FAILED")
            break
    else:
        print(f"   Ep {ep+1}: ❌ FAILED (timeout)")

env.close()

print(f"\n📊 Success Rate: {100*successes/10:.0f}%")

In [ ]:
# ==========================================
# CELL 11: RECORD VIDEO
# ==========================================

import imageio

print("🎥 Recording...")

env = MetaworldEnv(
    task=TASK_NAME,
    obs_type="pixels_agent_pos",
    render_mode="rgb_array",
    observation_width=480,
    observation_height=480,
)

obs, _ = env.reset(seed=42)
policy.reset()

frames = []
for step in range(300):
    frames.append(env.render())
    
    batch = {
        'observation.images.image': torch.from_numpy(obs['pixels']).float().permute(2,0,1).unsqueeze(0).to(device) / 255.0,
        'observation.state': torch.from_numpy(obs['agent_pos']).float().unsqueeze(0).to(device)
    }
    
    processed = preprocessor(batch)
    with torch.no_grad():
        action = policy.select_action(processed)
    action = postprocessor(action).cpu().numpy().squeeze()
    
    obs, _, term, trunc, info = env.step(action)
    if term or trunc or info.get('success', False):
        break

env.close()

imageio.mimsave('/kaggle/working/handlepull_eval.mp4', frames, fps=20)
print(f"🎥 Saved: /kaggle/working/handlepull_eval.mp4")

---
## ✅ Complete!

**Expected:** 60-80% success after 100k steps